# 원판을 얼마나, 어디까지 가리면 탐지가 달라질까?

같은 검증 프레임 31장에 같은 바닥 질감을 붙여 봅니다. 각 프레임에서 패치 폭은 고정하고 높이만 늘립니다. 매 단계에서 동일한 픽셀 조각을 원판 중앙과 아래쪽에 놓습니다. 0%에서는 두 위치가 같은 원본입니다.

**읽는 순서** · 첫 표에서 대상 프레임을 확인하고, 확대 그림에서 패치가 붙는 위치를 봅니다. 두 모델의 저장된 가중치로 추론한 뒤 전체 곡선과 이전 미탐 9장의 곡선을 읽습니다.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from ultralytics import YOLO

from wlcv.config import DATASET, RUNS
from wlcv.evaluation.occlusion_dose import (
    DOSES, POSITIONS, check_endpoints, count_frames, evaluate_doses,
    make_doses, mark_states, verify_scores,
)
from wlcv.evaluation.patch_position import make_example, paste_patch, reviewed_floor_frames

runs = {"Baseline": "plate_baseline", "Texture": "plate_occlusion_texture"}
report = RUNS / "plate_baseline/report"
out = report / "occlusion_dose"
out.mkdir(parents=True, exist_ok=True)
floors = reviewed_floor_frames(report / "occlusion_half_area/mask_positions.csv")
previous = pd.read_csv(report / "patch_position_comparison/per_frame.csv")

# 지난 실험에서 원본은 잡았고 아래 끝 또는 아래 윤곽 횡단에서 놓친 고유 프레임.
old_conditions = ["Lower strip", "Cross lower edge"]
prior = set(previous.loc[previous.lost & previous.condition.isin(old_conditions), "stem"])
assert len(floors) == 31 and len(prior) == 9 and prior.issubset(floors.index)
check_endpoints(floors, DATASET, previous)
display(floors.assign(previous_miss=floors.index.isin(prior))
        .groupby("person").agg(frames=("frame", "size"), previous_misses=("previous_miss", "sum")))

## 1. 패치와 원래 정답 박스 확인

파란 선은 합성 전 정답 박스입니다. 아래쪽 패치는 박스의 아래 변에 붙여 위로 키우고 중앙 패치는 같은 조각의 중심을 박스 중심에 맞춥니다. 작은 패치는 기존 25% 패치에서 아래쪽 행을 잘라 사용합니다. 그림에 적힌 값은 반올림한 픽셀 좌표로 계산한 실제 가림 면적입니다. 분모는 정답 박스 면적입니다.

In [ ]:
stem = "a03_sn_f_01_f19"  # 다른 프레임을 보려면 floors.index에서 선택하세요.
image, truth, _, donor, variants = make_example(DATASET, floors, stem)
placements, area = make_doses(truth, variants)
left, top, right, bottom = truth
bw, bh = right - left, bottom - top
fig, axes = plt.subplots(2, 4, figsize=(9, 5), layout="constrained")
for row, position in enumerate(POSITIONS):
    for col, dose in enumerate(DOSES):
        ax = axes[row, col]
        placement = placements.get((position, dose))
        ax.imshow(paste_patch(image, placement))
        ax.add_patch(Rectangle((left, top), bw, bh, fill=False, edgecolor="#00d9ff", linewidth=1))
        pct = 0
        if placement is not None:
            x, y, patch = placement
            height, width = patch.shape[:2]
            pct = 100 * width * height / area
        ax.set(xlim=(left - .18*bw, right + .18*bw), ylim=(bottom + .18*bh, top - .18*bh),
               title=f"{position} · {pct:.1f}%")
        ax.axis("off")
fig.suptitle(f"Plate crop · {stem}  |  Texture source · {donor}", fontsize=12)
fig.savefig(out / "patch_geometry.png", dpi=145, facecolor="white")
plt.show()

## 2. 같은 조건으로 두 모델 추론

Mac CPU, 입력 크기 640, 원판 클래스, 후보 신뢰도 하한 0.001을 사용합니다. **합성 전 정답 박스와 IoU가 0.50 이상인 후보 중 가장 높은 신뢰도**를 기록합니다. 후보가 있지만 신뢰도가 0.10 미만이면 저신뢰, 기준에 맞는 후보가 하나도 없으면 결측입니다. 두 모델의 0%·25% 신뢰도와 IoU도 지난 실행과 대조합니다.

In [ ]:
results = []
for name, run in runs.items():
    trained = pd.read_csv(RUNS / run / "train_people.csv")
    assert set(floors.person).isdisjoint(trained.person_id)
    model = YOLO(str((RUNS / run / "weights/best.pt").resolve(strict=True)))
    assert model.names == {0: "plate"}
    print(name)
    results.append(evaluate_doses(model, floors, DATASET).assign(model=name))

data = mark_states(pd.concat(results, ignore_index=True), prior)
verify_scores(data, previous)
data.to_csv(out / "per_frame.csv", index=False)
summary = count_frames(data)
summary.to_csv(out / "summary.csv")
display(summary)
display(count_frames(data[data.prior_miss]))

## 3. 전체 31장의 가림 비율별 신뢰도

한 선이 한 프레임입니다. 회색은 나머지 22장, 파란색과 주황색은 이전에 놓친 9장입니다. 점선은 정탐 신뢰도 0.10입니다. IoU 기준을 통과한 후보가 없을 때 선이 끊깁니다. 각 그래프 아래의 별도 줄은 해당 단계에서 매칭 후보가 없는 장수입니다.

In [ ]:
frame_colors = {"a03": "#187687", "b02": "#c0521c"}
fig = plt.figure(figsize=(10, 7.8), layout="constrained")
panels = fig.add_gridspec(2, 2)
for row, name in enumerate(runs):
    for col, position in enumerate(POSITIONS):
        sub = panels[row, col].subgridspec(2, 1, height_ratios=[5, 1])
        ax = fig.add_subplot(sub[0])
        counts = fig.add_subplot(sub[1], sharex=ax)
        view = data[(data.model == name) & (data.position == position)]
        for _, g in view.sort_values("prior_miss", kind="stable").groupby("stem", sort=False):
            g = g.sort_values("dose")
            highlight = g.prior_miss.iloc[0]
            color = frame_colors[g.person.iloc[0]] if highlight else "#aab0b7"
            ax.plot(g.dose, g.confidence, color=color, marker="o", markersize=3,
                    lw=1.2 if highlight else .8, alpha=.85 if highlight else .55)
        ax.axhline(.10, color="black", linestyle=":", lw=1)
        ax.set(title=f"{name} · {position}", xticks=DOSES, ylim=(-.025, 1.025), ylabel="Confidence")
        ax.tick_params(labelbottom=False)
        ax.grid(axis="y", alpha=.18)
        counts.set(ylim=(0, 1), yticks=[.5], yticklabels=["No match (n)"], xlabel="Occluded box area (%)")
        counts.tick_params(axis="y", length=0)
        for dose in DOSES:
            missing = view[view.dose.eq(dose)].confidence.isna().sum()
            counts.text(dose, .5, str(missing), ha="center", va="center")
handles = [Line2D([], [], color=c, lw=2, label=s) for s, c in
           [("Other 22 frames", "#aab0b7"), ("Previous misses · a03 (6)", frame_colors["a03"]),
            ("Previous misses · b02 (3)", frame_colors["b02"])]]
fig.legend(handles=handles, loc="outside upper center", ncols=3, frameon=False)
fig.savefig(out / "all_frames.png", dpi=150, facecolor="white")
plt.show()

## 4. 이전에 놓친 9장 확대

한 칸이 한 프레임입니다. 색은 모델, 선 모양은 위치입니다. 아래쪽 ×는 같은 색의 모델에서 매칭 후보가 없는 단계를 표시합니다. 아래 두 줄 C와 L은 각각 중앙과 아래쪽 조건입니다. ×끼리 겹치지 않도록 같은 줄에서 두 모델의 높이를 조금 벌렸습니다.

In [ ]:
colors = {"Baseline": "#1e638f", "Texture": "#c25720"}
styles = {"Center": "-", "Lower": "--"}
markers = {"Center": "o", "Lower": "^"}
focus = data[data.prior_miss]
frames = focus[["person", "frame", "stem"]].drop_duplicates().sort_values(["person", "frame"])
fig = plt.figure(figsize=(11, 9), layout="constrained")
panels = fig.add_gridspec(3, 3)
for i, item in enumerate(frames.itertuples()):
    sub = panels[i // 3, i % 3].subgridspec(2, 1, height_ratios=[4, 1])
    ax = fig.add_subplot(sub[0])
    missing = fig.add_subplot(sub[1], sharex=ax)
    for (name, position), g in focus[focus.stem == item.stem].groupby(["model", "position"]):
        g = g.sort_values("dose")
        ax.plot(g.dose, g.confidence, color=colors[name], ls=styles[position],
                marker=markers[position], ms=3.8, lw=1.5)
        absent = g[g.confidence.isna()]
        lane = (1 if position == "Center" else 0) + (.16 if name == "Baseline" else -.16)
        missing.scatter(absent.dose, [lane] * len(absent), c=colors[name], marker="x", s=35)
    ax.axhline(.10, color="black", ls=":", lw=.8)
    ax.set(title=f"{item.person} · frame {item.frame}", xticks=DOSES, ylim=(-.025, 1.025))
    ax.tick_params(labelbottom=False)
    ax.grid(axis="y", alpha=.18)
    missing.set(xlim=(-1.3, 26.3), ylim=(-.55, 1.55), yticks=[0, 1], yticklabels=["L", "C"])
    missing.set_facecolor("#f5f6f7")
    if i % 3 == 0:
        ax.set_ylabel("Confidence")
    if i >= 6:
        missing.set_xlabel("Occluded box area (%)")
handles = [Line2D([], [], color=colors[name], ls=styles[pos], marker=markers[pos], label=f"{name} · {pos}")
           for name in runs for pos in POSITIONS]
handles.append(Line2D([], [], color="black", ls="", marker="x", label="No IoU match"))
fig.legend(handles=handles, loc="outside upper center", ncols=5, frameon=False)
fig.savefig(out / "previous_nine.png", dpi=150, facecolor="white")
plt.show()

## 5. 같은 프레임에서 중앙과 아래쪽의 차이 확인

위치 한쪽만 정탐인 장수와, 두 위치 모두 후보가 있을 때의 신뢰도 차이(아래쪽 − 중앙)를 봅니다. 두 위치 중 한쪽이라도 후보가 없으면 신뢰도 차이 계산에서 빠지므로 유효 쌍 수도 함께 읽습니다. 프레임 표에서 `lost_match`는 바로 전 단계에 있던 매칭 후보가 사라졌음을, `regained_match`는 다시 나타났음을 뜻합니다.

In [ ]:
pairs = data.pivot(index=["model", "dose", "stem"], columns="position",
                   values=["confidence", "detected"])
scores = pairs["confidence"].astype(float)
hits = pairs["detected"].astype(bool)
paired = pd.DataFrame({
    "center_only": hits.Center & ~hits.Lower,
    "lower_only": hits.Lower & ~hits.Center,
    "difference": scores.Lower - scores.Center,
})
by_position = paired.groupby(["model", "dose"]).agg(
    center_only=("center_only", "sum"), lower_only=("lower_only", "sum"),
    pairs_with_scores=("difference", "count"), median_difference=("difference", "median"))
by_position.to_csv(out / "paired_positions.csv")
display(by_position.round(3))
example = focus[focus.stem.eq(stem)]
display(example[["stem", "model", "position", "dose", "confidence", "iou", "state"]].round(3))